# WindowScanOp Demo

This notebook demonstrates the `window_scan` operation and its helper functions:
- `window_scan` - The core operation for scanning windows and applying transforms
- `shuffle_scan` - Shuffle characters within each window
- `deletion_scan` - Delete (mark or remove) windows
- `insertion_scan` - Insert sequences at each position

The key innovation is support for **Pool-based transforms**, allowing you to use existing operations like `mutation_scan` as window transforms.


In [1]:
from poolparty.operations import (
    window_scan, shuffle_scan, deletion_scan, insertion_scan,
    mutation_scan, from_seqs
)
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)


## 1. Basic Window Scan with String Transforms

The simplest use case: apply a string transform to each window position.


In [2]:
# Reverse each 4-mer window as it scans across the sequence
pool = window_scan(
    background='ACGTACGT',
    window_size=4,
    transform=lambda w: w[::-1],  # Reverse the window
    mode='sequential'
)

print(f"Number of states: {pool.operation.num_states}")
print(f"Positions: {pool.operation.positions}")

df = pool.generate_library(num_complete_iterations=1)
df


Number of states: 5
Positions: [0, 1, 2, 3, 4]


,seq,op0(from_seqs).seq,op1(window_scan).seq,op1(window_scan).window_size,op1(window_scan).transformed_window,op1(window_scan).transform_idx,op1(window_scan).position,op1(window_scan).original_window
0,TGCAACGT,ACGTACGT,TGCAACGT,4,TGCA,0,0,ACGT
1,AATGCCGT,ACGTACGT,AATGCCGT,4,ATGC,0,1,CGTA
2,ACCATGGT,ACGTACGT,ACCATGGT,4,CATG,0,2,GTAC
3,ACGGCATT,ACGTACGT,ACGGCATT,4,GCAT,0,3,TACG
4,ACGTTGCA,ACGTACGT,ACGTTGCA,4,TGCA,0,4,ACGT


## 2. Deletion Scan

Scan windows and delete them - either marking with dashes (preserving length) or actually removing them.


In [8]:
# Marked deletion (replace with dashes, keeps sequence length)
pool = deletion_scan(
    background='ACAGCTGACAA',
    deletion_size=4,
    mark_deletions=True,
    mode='sequential'
)

df = pool.generate_library(num_complete_iterations=1)
print("Marked deletions (length preserved):")
for seq in df['seq']:
    print(f"  {seq}")


Marked deletions (length preserved):
  ----CTGACAA
  A----TGACAA
  AC----GACAA
  ACA----ACAA
  ACAG----CAA
  ACAGC----AA
  ACAGCT----A
  ACAGCTG----


In [7]:
# Actual deletion (removes characters, changes sequence length)
pool = deletion_scan(
    background='ACAGCTGACAA',
    deletion_size=4,
    mark_deletions=False,
    mode='sequential'
)

df = pool.generate_library(num_complete_iterations=1)
print("Actual deletions (length changes):")
for seq in df['seq']:
    print(f"  {seq} (length: {len(seq)})")


Actual deletions (length changes):
  CTGACAA (length: 7)
  ATGACAA (length: 7)
  ACGACAA (length: 7)
  ACAACAA (length: 7)
  ACAGCAA (length: 7)
  ACAGCAA (length: 7)
  ACAGCTA (length: 7)
  ACAGCTG (length: 7)


## 3. Insertion Scan

Insert a sequence at each position - either overwriting existing characters or inserting between positions.


In [9]:
# Overwrite mode (replaces existing characters)
pool = insertion_scan(
    background='AAAAAAAA',
    insert='GGG',
    overwrite=True,
    mode='sequential'
)

df = pool.generate_library(num_complete_iterations=1)
print("Overwrite mode (length preserved):")
for seq in df['seq']:
    print(f"  {seq}")


Overwrite mode (length preserved):
  GGGAAAAA
  AGGGAAAA
  AAGGGAAA
  AAAGGGAA
  AAAAGGGA
  AAAAAGGG


In [10]:
# Insert mode (inserts between positions, expands sequence)
pool = insertion_scan(
    background='AAAA',
    insert='GGG',
    overwrite=False,
    mode='sequential'
)

df = pool.generate_library(num_complete_iterations=1)
print("Insert mode (sequence expands):")
for seq in df['seq']:
    print(f"  {seq} (length: {len(seq)})")


Insert mode (sequence expands):
  GGGAAAA (length: 7)
  AGGGAAA (length: 7)
  AAGGGAA (length: 7)
  AAAGGGA (length: 7)
  AAAAGGG (length: 7)


## 4. Shuffle Scan

Shuffle the characters within each window position.


In [11]:
# Shuffle 4-mer windows
pool = shuffle_scan(
    background='ACGTACGT',
    shuffle_size=4,
    seed=42,
    mode='sequential'
)

df = pool.generate_library(num_complete_iterations=1)
print("Shuffled windows:")
for i, row in df.iterrows():
    pos_col = [c for c in df.columns if 'position' in c][0]
    print(f"  Position {row[pos_col]}: {row['seq']}")


Shuffled windows:
  Position 0: TGCAACGT
  Position 1: AATCGCGT
  Position 2: ACGCATGT
  Position 3: ACGCGTAT
  Position 4: ACGTGTAC


## 5. Pool-Based Transforms (Advanced)

The real power of `window_scan` is the ability to use **existing Pool operations** as transforms.

This creates a single transform Pool instance that is reused for all windows, enabling efficient batch execution.


In [12]:
# Use mutation_scan to mutate within each window
# The transform receives a "slot" Pool and returns a new Pool
pool = window_scan(
    background='AAAAAAAA',
    window_size=4,
    transform=lambda slot: mutation_scan(slot, num_mutations=1, mode='sequential'),
    mode='sequential'
)

print(f"Number of states: {pool.operation.num_states}")
print(f"  = {len(pool.operation.positions)} positions × 12 mutation states")
print(f"  (12 = 4 positions × 3 mutation choices per position)")


Number of states: 60
  = 5 positions × 12 mutation states
  (12 = 4 positions × 3 mutation choices per position)


In [13]:
# Generate first 24 sequences (first 2 positions, all 12 mutations each)
df = pool.generate_library(num_seqs=24, init_state=0)

# Show results grouped by position
pos_col = [c for c in df.columns if 'position' in c and 'transform' not in c][0]

for pos in df[pos_col].unique():
    print(f"\nPosition {pos}:")
    pos_df = df[df[pos_col] == pos]
    for seq in pos_df['seq'].head(6):  # Show first 6
        print(f"  {seq}")
    if len(pos_df) > 6:
        print(f"  ... and {len(pos_df) - 6} more")



Position 0:
  CAAAAAAA
  GAAAAAAA
  TAAAAAAA
  ACAAAAAA
  AGAAAAAA
  ATAAAAAA
  ... and 6 more

Position 1:
  ACAAAAAA
  AGAAAAAA
  ATAAAAAA
  AACAAAAA
  AAGAAAAA
  AATAAAAA
  ... and 6 more


## 6. Design Cards from Pool Transforms

When using Pool-based transforms, the design cards from the transform operation are captured and prefixed with `transform.`


In [14]:
# Generate with Pool-based transform
pool = window_scan(
    background='AAAA',
    window_size=4,
    transform=lambda slot: mutation_scan(slot, num_mutations=1, mode='sequential'),
    mode='sequential'
)

df = pool.generate_library(num_seqs=12, init_state=0)

# Show all columns
print("All design card columns:")
for col in df.columns:
    print(f"  {col}")


All design card columns:
  seq
  op28(from_seqs).seq
  op35(window_scan).seq
  op35(window_scan).window_size
  op35(window_scan).transformed_window
  op35(window_scan).transform_idx
  op35(window_scan).position
  op35(window_scan).original_window
  op35(window_scan).transform.op33(slot).seq
  op35(window_scan).transform.op33(slot).slot_index
  op35(window_scan).transform.op34(mutation_scan).seq
  op35(window_scan).transform.op34(mutation_scan).mut_chars
  op35(window_scan).transform.op34(mutation_scan).wt_chars
  op35(window_scan).transform.op34(mutation_scan).positions


In [15]:
# Focus on the transform-related columns
transform_cols = ['seq'] + [c for c in df.columns if 'transform.' in c]
df[transform_cols].head(12)


,seq,op35(window_scan).transform.op33(slot).seq,op35(window_scan).transform.op33(slot).slot_index,op35(window_scan).transform.op34(mutation_scan).seq,op35(window_scan).transform.op34(mutation_scan).mut_chars,op35(window_scan).transform.op34(mutation_scan).wt_chars,op35(window_scan).transform.op34(mutation_scan).positions
0,CAAA,AAAA,0,CAAA,"(C,)","(A,)","(0,)"
1,GAAA,AAAA,1,GAAA,"(G,)","(A,)","(0,)"
2,TAAA,AAAA,2,TAAA,"(T,)","(A,)","(0,)"
3,ACAA,AAAA,3,ACAA,"(C,)","(A,)","(1,)"
4,AGAA,AAAA,4,AGAA,"(G,)","(A,)","(1,)"
5,ATAA,AAAA,5,ATAA,"(T,)","(A,)","(1,)"
6,AACA,AAAA,6,AACA,"(C,)","(A,)","(2,)"
7,AAGA,AAAA,7,AAGA,"(G,)","(A,)","(2,)"
8,AATA,AAAA,8,AATA,"(T,)","(A,)","(2,)"
9,AAAC,AAAA,9,AAAC,"(C,)","(A,)","(3,)"


## Summary

The `window_scan` operation provides a flexible framework for:

1. **Simple string transforms** - reverse, lowercase, replace, etc.
2. **Deletions** - marked or actual removal of windows
3. **Insertions** - overwrite or insert mode
4. **Shuffles** - randomize characters within windows
5. **Pool-based transforms** - use any existing operation as a window transform

Key features:
- Single transform Pool instance (efficient batch execution)
- Design card propagation from transform operations
- Flexible position specification (range or explicit)
- Multiple transforms per window for random sampling
- Both sequential and random modes
